### Training Model

### RAED-VP — BDD100K MOT test (quatre classes)

In [ ]:
%pip install -q ultralytics

Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import time
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO
from IPython.display import display


# ============================================================
# 1. PATHS
# ============================================================

DATA_ROOT = Path(
    "/kaggle/working/bdd100k_test"
)

MODEL_PATH = (
    "/kaggle/input/models/chaymaerami/testmodelyolo/other/default/1/test.pt"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "raed_bdd_multiclass_validation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# 2. EVALUATED CLASSES
# ============================================================

# Actual YOLO checkpoint class names:
# {0:'class0', 1:'class1', 2:'class2', 3:'class3'}
# IMPORTANT: The semantic meaning of these IDs MUST match the
# data.yaml used at training time (see below).

SELECTED_CLASS_IDS = [
    0,
    1,
    2,
    3
]

EVAL_CLASSES = [
    "class0",
    "class1",
    "class2",
    "class3"
]

# IMPORTANT: verify against the training data.yaml.
# Assumption: 0=pedestrian, 1=car, 2=bus, 3=truck.
GT_CLASS_MAP = {
    "pedestrian": "class0",
    "car": "class1",
    "bus": "class2",
    "truck": "class3",
}

# All four evaluation categories refer to the generic YOLO model names.
assert set(GT_CLASS_MAP.values()) == set(EVAL_CLASSES)


# ============================================================
# 3. EXPERIMENT CONFIGURATION
# ============================================================

ANNOTATION_FPS = 5.0

SYNC_MODE = "six"  # "six" or "ratio"
SYNC_OFFSET = 0


INFERENCE_CONF = 0.001

DECISION_CONF = 0.30

NMS_IOU = 0.60

MAX_DET = 300


DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)



MAX_VIDEOS = None

MAX_FRAMES_PER_VIDEO = None

FIXED_DELTA_REF = 0.0157

FIXED_ALPHA = 0.50
FIXED_BETA  = 0.40
FIXED_GAMMA = 0.10

FIXED_KMIN = 1
FIXED_KMAX = 3

NMAX_DENSITY = 20
U0 = 0.50


# Event baseline
# UA-DETRAC-calibrated event baseline threshold (Eq. 24 manuscript).
FIXED_EVENT_THETA = 0.0140
FIXED_EVENT_MAX_SKIP = 3


# Fixed stride baseline
FIXED_STRIDE = 3


print("=" * 75)
print("EXPERIMENT CONFIGURATION")
print("=" * 75)

print("Device            :", DEVICE)
print("Evaluated classes :", EVAL_CLASSES)
print("Class IDs         :", SELECTED_CLASS_IDS)
print("Inference conf    :", INFERENCE_CONF)
print("Decision conf     :", DECISION_CONF)
print("NMS IoU           :", NMS_IOU)
print("Annotation FPS    :", ANNOTATION_FPS)
print("Synchronization   :", SYNC_MODE, "+", SYNC_OFFSET)
print("Ground-truth map  :", GT_CLASS_MAP)
print("MAX_VIDEOS        :", MAX_VIDEOS)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
EXPERIMENT CONFIGURATION
Device            : 0
Evaluated classes : ['class0', 'class1', 'class2', 'class3']
Class IDs         : [0, 1, 2, 3]
Inference conf    : 0.001
Decision conf     : 0.3
NMS IoU           : 0.6
Annotation FPS    : 5.0
Synchronization   : six + 0
Ground-truth map  : {'pedestrian': 'class0', 'car': 'class1', 'bus': 'class2', 'truck': 'class3'}
MAX_VIDEOS        : None


In [3]:
# ============================================================
# 5. FIND VIDEO AND ANNOTATION FILES
# ============================================================

VIDEO_EXTS = [
    ".mp4",
    ".avi",
    ".mov",
    ".mkv"
]

video_files = []

for ext in VIDEO_EXTS:

    video_files.extend(
        DATA_ROOT.rglob(
            f"*{ext}"
        )
    )

video_files = sorted(
    video_files
)


# ONLY the MOT annotation CSV. Loading split manifests as annotation CSVs
# would silently contaminate / fail the evaluation.
csv_files = [DATA_ROOT / "mot_labels.csv"]
if not csv_files[0].is_file():
    raise FileNotFoundError(
        f"mot_labels.csv introuvable : {csv_files[0]}"
    )


print("\nVideos found :", len(video_files))
print("CSV files    :", len(csv_files))


if len(video_files) == 0:

    raise RuntimeError(
        "No video file found. Check DATA_ROOT."
    )


if len(csv_files) == 0:

    raise RuntimeError(
        "No CSV annotation file found."
    )


# ============================================================
# 6. LOAD ALL CSV ANNOTATIONS
# ============================================================

def normalize_col_name(col):

    return (
        str(col)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


raw_dfs = []


for csv_path in csv_files:

    try:

        temp = pd.read_csv(
            csv_path
        )

        temp.columns = [
            normalize_col_name(c)
            for c in temp.columns
        ]

        temp["__ann_file__"] = str(
            csv_path
        )

        raw_dfs.append(
            temp
        )

    except Exception as e:

        print(
            "Could not read:",
            csv_path,
            e
        )


if len(raw_dfs) == 0:

    raise RuntimeError(
        "No annotation CSV could be loaded."
    )


raw_ann = pd.concat(
    raw_dfs,
    ignore_index=True
)


print(
    "\nRaw annotation rows:",
    len(raw_ann)
)

print(
    "Raw columns:",
    list(raw_ann.columns)
)


# ============================================================
# 7. STANDARDIZE ANNOTATIONS
# ============================================================

required_columns = [
    "videoname",
    "frameindex",
    "category",
    "box2d.x1",
    "box2d.x2",
    "box2d.y1",
    "box2d.y2"
]


missing_columns = [
    c
    for c in required_columns
    if c not in raw_ann.columns
]


if missing_columns:

    raise ValueError(
        "Missing required annotation columns: "
        + str(missing_columns)
    )


# Annotated timestamps (including frames without valid target boxes).
# Preserve them BEFORE dropping rows with missing bounding-box coordinates.
ann_timestamps = raw_ann[["videoname", "frameindex"]].copy()
ann_timestamps["videoname"] = ann_timestamps["videoname"].astype(str).str.strip()
ann_timestamps["frameindex"] = pd.to_numeric(
    ann_timestamps["frameindex"], errors="coerce"
)
ann_timestamps = ann_timestamps.dropna(subset=["frameindex"])
ann_timestamps["frameindex"] = ann_timestamps["frameindex"].astype(int)

ann = pd.DataFrame()


ann["video_key"] = (
    raw_ann["videoname"]
    .astype(str)
    .str.strip()
)


ann["frame_id"] = pd.to_numeric(
    raw_ann["frameindex"],
    errors="coerce"
)


ann["category"] = (
    raw_ann["category"]
    .astype(str)
    .str.lower()
    .str.strip()
)


ann["x1"] = pd.to_numeric(
    raw_ann["box2d.x1"],
    errors="coerce"
)

ann["y1"] = pd.to_numeric(
    raw_ann["box2d.y1"],
    errors="coerce"
)

ann["x2"] = pd.to_numeric(
    raw_ann["box2d.x2"],
    errors="coerce"
)

ann["y2"] = pd.to_numeric(
    raw_ann["box2d.y2"],
    errors="coerce"
)


ann = ann.dropna(
    subset=[
        "video_key",
        "frame_id",
        "category",
        "x1",
        "y1",
        "x2",
        "y2"
    ]
).copy()


ann["frame_id"] = (
    ann["frame_id"]
    .astype(int)
)


# valid boxes only
ann = ann[
    ann["x2"] > ann["x1"]
].copy()

ann = ann[
    ann["y2"] > ann["y1"]
].copy()


print(
    "\nStandard annotation rows:",
    len(ann)
)


print(
    "\nALL DATASET CATEGORIES:"
)

print(
    ann["category"]
    .value_counts()
)



Videos found : 50
CSV files    : 1

Raw annotation rows: 106003
Raw columns: ['name', 'videoname', 'frameindex', 'id', 'category', 'attributes.crowd', 'attributes.occluded', 'attributes.truncated', 'box2d.x1', 'box2d.x2', 'box2d.y1', 'box2d.y2', 'havevideo', '__ann_file__']

Standard annotation rows: 105734

ALL DATASET CATEGORIES:
category
car              75851
pedestrian       16393
truck             6180
bus               2994
bicycle           1807
rider             1298
motorcycle         707
other vehicle      329
other person       107
trailer             68
Name: count, dtype: int64


In [4]:
# ============================================================
# 8. FILTER STATISTICS FOR FOUR SELECTED CLASSES
# ============================================================

# IMPORTANT: ann['category'] contains the ORIGINAL BDD100K names,
# whereas EVAL_CLASSES contains the model's generic names.
selected_ann = ann[
    ann["category"].isin(GT_CLASS_MAP.keys())
].copy()
selected_ann["eval_class"] = selected_ann["category"].map(GT_CLASS_MAP)

if selected_ann.empty:
    raise RuntimeError(
        "Aucune GT sélectionnée ! Vérifiez GT_CLASS_MAP et le CSV. "
        f"Exemples de catégories : {ann['category'].unique()[:20]}"
    )


print(
    "\nSELECTED FOUR-CLASS GT:"
)

print(
    selected_ann[
        "category"
    ]
    .value_counts()
)


print(
    "\nSelected GT instances:",
    len(selected_ann)
)


# ============================================================
# 9. MATCH VIDEO FILES TO ANNOTATIONS
# ============================================================

available_ann_keys = set(ann_timestamps["videoname"].unique())


def match_video_to_ann_key(video_stem, ann_keys):
    """One-to-one matching: no substring matching, which can assign wrong GT."""
    key_map = {str(k).lower().strip(): k for k in ann_keys}
    return key_map.get(str(video_stem).lower().strip())

video_df = pd.DataFrame({

    "video_path": [
        str(p)
        for p in video_files
    ],

    "video_key": [
        p.stem
        for p in video_files
    ]
})


video_df["ann_key"] = (
    video_df[
        "video_key"
    ]
    .apply(
        lambda x:
        match_video_to_ann_key(
            x,
            available_ann_keys
        )
    )
)


video_df["has_annotation"] = (
    video_df[
        "ann_key"
    ]
    .notna()
)


valid_videos = (
    video_df[
        video_df[
            "has_annotation"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


if MAX_VIDEOS is not None:

    valid_videos = (
        valid_videos
        .head(
            MAX_VIDEOS
        )
        .copy()
    )


print("\nMatched videos:", len(valid_videos))
print("Unmatched videos:", len(video_df) - len(valid_videos))
if len(valid_videos) < len(video_df):
    print(video_df.loc[~video_df["has_annotation"], "video_key"].head(10).tolist())
    if MAX_VIDEOS is None:
        raise RuntimeError(
            "Des vidéos test n'ont pas de correspondance exacte dans mot_labels.csv."
        )


if len(valid_videos) == 0:

    raise RuntimeError(
        "No video matched annotation keys."
    )



SELECTED FOUR-CLASS GT:
category
car           75851
pedestrian    16393
truck          6180
bus            2994
Name: count, dtype: int64

Selected GT instances: 101418

Matched videos: 50
Unmatched videos: 0


In [5]:
# ============================================================
# 10. LOAD MODEL
# ============================================================

model = YOLO(
    MODEL_PATH
)


print(
    "\nMODEL CLASSES:"
)

print(
    model.names
)


# ============================================================
# 11. VERIFY SELECTED MODEL CLASSES
# ============================================================

for class_id in SELECTED_CLASS_IDS:

    if class_id not in model.names:

        raise ValueError(
            f"Class ID {class_id} "
            "not present in model.names"
        )


selected_model_names = {

    class_id:
        str(
            model.names[
                class_id
            ]
        )
        .lower()
        .strip()

    for class_id
    in SELECTED_CLASS_IDS
}


print(
    "\nSELECTED MODEL CLASSES:"
)

print(
    selected_model_names
)


expected_mapping = {
    0: "class0",
    1: "class1",
    2: "class2",
    3: "class3"
}


for cid, expected_name in (
    expected_mapping.items()
):

    actual_name = (
        selected_model_names[
            cid
        ]
    )

    if actual_name != expected_name:

        raise ValueError(
            f"Class mismatch: ID {cid} = "
            f"'{actual_name}', expected "
            f"'{expected_name}'"
        )


# ============================================================
# 12. IoU
# ============================================================

def box_iou(
    box_a,
    box_b
):

    ix1 = max(
        float(box_a[0]),
        float(box_b[0])
    )

    iy1 = max(
        float(box_a[1]),
        float(box_b[1])
    )

    ix2 = min(
        float(box_a[2]),
        float(box_b[2])
    )

    iy2 = min(
        float(box_a[3]),
        float(box_b[3])
    )


    iw = max(
        0.0,
        ix2 - ix1
    )

    ih = max(
        0.0,
        iy2 - iy1
    )


    intersection = (
        iw * ih
    )


    area_a = (
        max(
            0.0,
            float(box_a[2])
            -
            float(box_a[0])
        )

        *

        max(
            0.0,
            float(box_a[3])
            -
            float(box_a[1])
        )
    )


    area_b = (
        max(
            0.0,
            float(box_b[2])
            -
            float(box_b[0])
        )

        *

        max(
            0.0,
            float(box_b[3])
            -
            float(box_b[1])
        )
    )


    union = (
        area_a
        +
        area_b
        -
        intersection
    )


    if union <= 0:

        return 0.0


    return float(
        intersection
        /
        union
    )


# ============================================================
# 13. YOLO MULTI-CLASS INFERENCE
# ============================================================
#
# Prediction format:
#
# [
#   x1,
#   y1,
#   x2,
#   y2,
#   confidence,
#   class_name
# ]
#
# ============================================================

def yolo_predict_frame(
    frame
):

    start = time.perf_counter()


    results = model.predict(

        source=frame,
        imgsz=768 ,

        conf=
            INFERENCE_CONF,

        iou=
            NMS_IOU,

        classes=
            SELECTED_CLASS_IDS,

        max_det=
            MAX_DET,

        verbose=False,

        device=
            DEVICE
    )


    latency_ms = (
        time.perf_counter()
        -
        start
    ) * 1000.0


    predictions = []


    result = results[0]


    if (
        result.boxes is None
        or
        len(result.boxes) == 0
    ):

        return (
            predictions,
            latency_ms
        )


    boxes = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )


    confidences = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
    )


    class_ids = (
        result.boxes.cls
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )


    for (
        box,
        confidence,
        class_id
    ) in zip(
        boxes,
        confidences,
        class_ids
    ):

        class_name = (
            str(
                model.names[
                    int(class_id)
                ]
            )
            .lower()
            .strip()
        )


        if (
            class_name
            not in EVAL_CLASSES
        ):

            continue


        x1, y1, x2, y2 = (
            box.tolist()
        )


        predictions.append([

            float(x1),
            float(y1),
            float(x2),
            float(y2),

            float(confidence),

            class_name
        ])


    return (
        predictions,
        latency_ms
    )


# ============================================================
# 14. VISUAL CHANGE
# ============================================================

def compute_visual_delta(
    previous_frame,
    current_frame,
    output_size=(160, 90)
):

    if (
        previous_frame is None
        or
        current_frame is None
    ):

        return 1.0


    previous_gray = (
        cv2.cvtColor(
            previous_frame,
            cv2.COLOR_BGR2GRAY
        )
    )


    current_gray = (
        cv2.cvtColor(
            current_frame,
            cv2.COLOR_BGR2GRAY
        )
    )


    previous_small = (
        cv2.resize(
            previous_gray,
            output_size
        )
    )


    current_small = (
        cv2.resize(
            current_gray,
            output_size
        )
    )


    difference = np.abs(

        current_small.astype(
            np.float32
        )

        -

        previous_small.astype(
            np.float32
        )
    )


    return float(
        difference.mean()
        /
        255.0
    )


# ============================================================
# 15. GET VIDEO GT
# ============================================================
#
# GT format:
#
# [
#   x1,
#   y1,
#   x2,
#   y2,
#   class_name
# ]
#
# ============================================================

def get_video_annotation_info(
    annotation_df,
    annotation_key
):

    df_all = (
        annotation_df[
            annotation_df[
                "video_key"
            ].astype(str)
            ==
            str(annotation_key)
        ]
        .copy()
    )


    timestamp_ids = sorted(
        ann_timestamps.loc[
            ann_timestamps["videoname"] == str(annotation_key),
            "frameindex"
        ].unique().tolist()
    )

    if not timestamp_ids:
        return {}, None, None, []


    min_frame = int(min(timestamp_ids))
    max_frame = int(max(timestamp_ids))
    labeled_ids = [int(i) for i in timestamp_ids]


    # --------------------------------------------------------
    # Four selected categories only
    # --------------------------------------------------------

    df_selected = (
        df_all[df_all["category"].isin(GT_CLASS_MAP.keys())]
        .copy()
    )


    gt_by_frame = {}


    for frame_id, group in (
        df_selected
        .groupby(
            "frame_id"
        )
    ):

        objects = []


        for _, row in (
            group.iterrows()
        ):

            class_name = (
                str(
                    row[
                        "category"
                    ]
                )
                .lower()
                .strip()
            )


            if (
                class_name
                not in GT_CLASS_MAP
            ):

                continue


            mapped_class = (
                GT_CLASS_MAP[
                    class_name
                ]
            )


            objects.append([

                float(
                    row["x1"]
                ),

                float(
                    row["y1"]
                ),

                float(
                    row["x2"]
                ),

                float(
                    row["y2"]
                ),

                mapped_class
            ])


        gt_by_frame[
            int(frame_id)
        ] = objects


    return (
        gt_by_frame,
        min_frame,
        max_frame,
        labeled_ids
    )



MODEL CLASSES:
{0: 'class0', 1: 'class1', 2: 'class2', 3: 'class3'}

SELECTED MODEL CLASSES:
{0: 'class0', 1: 'class1', 2: 'class2', 3: 'class3'}


In [6]:
# ============================================================
# 16. ANNOTATION 5 FPS -> RAW VIDEO ~30 FPS
# ============================================================
def build_annotation_mapping(video_fps, total_raw_frames, labeled_ann_ids):
    """Return raw-frame-index -> annotation-frame-index, using tested mode.

    Do NOT clamp overflowing indices to the last frame (causes incorrect labels).
    """
    if SYNC_MODE == "six":
        factor = 6.0
    elif SYNC_MODE == "ratio":
        factor = float(video_fps) / float(ANNOTATION_FPS)
    else:
        raise ValueError(f"SYNC_MODE invalide : {SYNC_MODE!r}")

    raw_to_annotation = {}
    dropped = []
    for ann_id in labeled_ann_ids:
        raw_idx = int(round(int(ann_id) * factor)) + int(SYNC_OFFSET)
        if raw_idx < 0 or raw_idx >= total_raw_frames:
            dropped.append((ann_id, raw_idx))
            continue
        if raw_idx in raw_to_annotation:
            raise RuntimeError(
                f"Collision de mapping à raw_idx={raw_idx}; "
                "vérifier ANNOTATION_FPS et SYNC_MODE."
            )
        raw_to_annotation[raw_idx] = int(ann_id)

    if len(raw_to_annotation) == 0:
        raise RuntimeError(
            f"Aucune frame annotée synchronisée (mode={SYNC_MODE}, "
            f"offset={SYNC_OFFSET}, fps={video_fps}, nb_frames={total_raw_frames})."
        )
    if dropped:
        print(f"AVERTISSEMENT: {len(dropped)} annotations hors vidéo, exemples: {dropped[:3]}")
    return raw_to_annotation, factor


# ============================================================
# 17. ALIGNMENT CHECK ON FIRST VIDEO
# ============================================================

first_video_row = (
    valid_videos.iloc[0]
)


test_video_path = (
    first_video_row[
        "video_path"
    ]
)


test_ann_key = (
    first_video_row[
        "ann_key"
    ]
)


test_cap = cv2.VideoCapture(
    str(
        test_video_path
    )
)


test_total_frames = int(
    test_cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)


test_fps = float(
    test_cap.get(
        cv2.CAP_PROP_FPS
    )
)


test_cap.release()


(
    test_gt,
    test_min_ann,
    test_max_ann,
    test_labeled_ids
) = get_video_annotation_info(

    ann,
    test_ann_key
)


(
    test_mapping,
    test_ratio
) = build_annotation_mapping(

    test_fps,

    test_total_frames,

    test_labeled_ids
)


print(
    "\n" + "=" * 75
)

print(
    "TEMPORAL ALIGNMENT CHECK"
)

print(
    "=" * 75
)


print(
    "Video:",
    Path(
        test_video_path
    ).stem
)

print(
    "Video FPS:",
    test_fps
)

print(
    "Raw frames:",
    test_total_frames
)

print(
    "Annotation min index:",
    test_min_ann
)

print(
    "Annotation max index:",
    test_max_ann
)

print(
    "Unique annotation frames:",
    len(
        test_labeled_ids
    )
)

print(
    "Raw/annotation ratio:",
    test_ratio
)

print(
    "Mapped evaluation timestamps:",
    len(
        test_mapping
    )
)


print(
    "\nFirst annotation mappings:"
)


for raw_index in (
    sorted(
        test_mapping.keys()
    )[:10]
):

    print(
        f"ann "
        f"{test_mapping[raw_index]:3d}"
        f" -> raw "
        f"{raw_index:4d}"
    )



# ============================================================
# 17b. ASSERT CLASS MAPPING BEFORE EXPENSIVE INFERENCE
# ============================================================
probe_n_gt = 0
for _, probe_video in valid_videos.iterrows():
    probe_gt, _, _, probe_ann_indices = get_video_annotation_info(
        ann, probe_video["ann_key"]
    )
    probe_n_gt = sum(len(objects) for objects in probe_gt.values())
    if probe_n_gt > 0:
        break
print("Précontrôle: GT sélectionnées dans une vidéo:", probe_n_gt)
if probe_n_gt == 0:
    raise RuntimeError(
        "Aucun objet GT pour les classes cibles ! Contrôler GT_CLASS_MAP."
    )
print("Précontrôle: GT OK.")


AVERTISSEMENT: 2 annotations hors vidéo, exemples: [(201, 1206), (202, 1212)]

TEMPORAL ALIGNMENT CHECK
Video: 001c5339-9a6cdd3e
Video FPS: 29.930452061599603
Raw frames: 1205
Annotation min index: 0
Annotation max index: 202
Unique annotation frames: 203
Raw/annotation ratio: 6.0
Mapped evaluation timestamps: 201

First annotation mappings:
ann   0 -> raw    0
ann   1 -> raw    6
ann   2 -> raw   12
ann   3 -> raw   18
ann   4 -> raw   24
ann   5 -> raw   30
ann   6 -> raw   36
ann   7 -> raw   42
ann   8 -> raw   48
ann   9 -> raw   54
Précontrôle: GT sélectionnées dans une vidéo: 1878
Précontrôle: GT OK.


In [7]:
# ============================================================
# 18. BUILD CONTINUOUS CACHE
# ============================================================

def build_continuous_cache_for_video(
    video_path,
    annotation_key
):

    (
        gt_by_ann_frame,
        min_ann_frame,
        max_ann_frame,
        labeled_ids
    ) = get_video_annotation_info(

        ann,
        annotation_key
    )


    capture = (
        cv2.VideoCapture(
            str(
                video_path
            )
        )
    )


    if not capture.isOpened():

        raise RuntimeError(
            f"Cannot open video: "
            f"{video_path}"
        )


    total_raw_frames = int(
        capture.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )


    video_fps = float(
        capture.get(
            cv2.CAP_PROP_FPS
        )
    )


    (
        raw_to_ann,
        mapping_ratio
    ) = build_annotation_mapping(

        video_fps,

        total_raw_frames,

        labeled_ids
    )


    records = []


    previous_frame = None

    raw_frame_idx = 0


    progress = tqdm(

        total=
            total_raw_frames,

        desc=
            Path(
                video_path
            ).name,

        leave=False
    )


    while True:

        success, frame = (
            capture.read()
        )


        if not success:

            break


        # ----------------------------------------------------
        # Is this raw frame one of the 5-FPS
        # annotation timestamps?
        # ----------------------------------------------------

        ann_frame_idx = (
            raw_to_ann.get(
                raw_frame_idx,
                None
            )
        )


        is_labeled = (
            ann_frame_idx
            is not None
        )


        if is_labeled:

            # [] means:
            # labeled timestamp but zero objects
            # among our four selected classes.
            gt_objects = (
                gt_by_ann_frame.get(
                    ann_frame_idx,
                    []
                )
            )

        else:

            # None means:
            # raw stream frame without tracking
            # annotation timestamp.
            gt_objects = None


        # ----------------------------------------------------
        # Visual difference on ALL stream frames
        # ----------------------------------------------------

        delta = (
            compute_visual_delta(
                previous_frame,
                frame
            )
        )


        # ----------------------------------------------------
        # Continuous detector output
        # ----------------------------------------------------

        predictions, latency_ms = (
            yolo_predict_frame(
                frame
            )
        )


        records.append({

            "video":
                Path(
                    video_path
                ).stem,

            "raw_frame_idx":
                raw_frame_idx,

            "ann_frame_idx":
                ann_frame_idx,

            "is_labeled":
                is_labeled,

            "gt_objects":
                gt_objects,

            "continuous_preds":
                predictions,

            "continuous_latency_ms":
                float(
                    latency_ms
                ),

            "delta":
                float(
                    delta
                ),

            "video_fps":
                float(
                    video_fps
                ),

            "mapping_ratio":
                float(
                    mapping_ratio
                )
        })


        previous_frame = frame

        raw_frame_idx += 1


        progress.update(1)


        if (
            MAX_FRAMES_PER_VIDEO
            is not None
            and
            raw_frame_idx
            >=
            MAX_FRAMES_PER_VIDEO
        ):

            break


    progress.close()

    capture.release()


    return records


# ============================================================
# 19. RUN CONTINUOUS INFERENCE
# ============================================================

all_records = []


print(
    "\n" + "=" * 75
)

print(
    "RUNNING CONTINUOUS DETECTOR CACHE"
)

print(
    "=" * 75
)


for _, video_row in tqdm(

    valid_videos.iterrows(),

    total=
        len(
            valid_videos
        ),

    desc="Videos"
):

    video_records = (
        build_continuous_cache_for_video(

            video_row[
                "video_path"
            ],

            video_row[
                "ann_key"
            ]
        )
    )


    all_records.extend(
        video_records
    )


print(
    "\nTotal stream frames:",
    len(
        all_records
    )
)


n_eval_frames = sum(

    int(
        r[
            "gt_objects"
        ]
        is not None
    )

    for r
    in all_records
)


n_gt_objects = sum(

    len(
        r[
            "gt_objects"
        ]
    )

    for r
    in all_records

    if (
        r[
            "gt_objects"
        ]
        is not None
    )
)


print(
    "Annotated evaluation frames:",
    n_eval_frames
)

print("Four-class GT objects:", n_gt_objects)

if n_eval_frames == 0 or n_gt_objects == 0:
    raise RuntimeError(
        "Évaluation impossible : aucune frame annotée ou aucun objet GT. "
        "Vérifiez le mapping des classes, les identifiants vidéo et SYNC_MODE."
    )

# If the full test set contains four categories, each should have GT > 0.
from collections import Counter
actual_gt_distribution = Counter(
    obj[4]
    for r in all_records
    if r["gt_objects"] is not None
    for obj in r["gt_objects"]
)
print("Four-class GT distribution:", dict(actual_gt_distribution))
if MAX_VIDEOS is None and MAX_FRAMES_PER_VIDEO is None:
    absent = [x for x in EVAL_CLASSES if actual_gt_distribution[x] == 0]
    if absent:
        raise RuntimeError(
            f"Classes sans GT sur le test complet: {absent}. "
            "Ne pas publier de mAP avec des classes absentes."
        )



RUNNING CONTINUOUS DETECTOR CACHE


Videos:   0%|          | 0/50 [00:00<?, ?it/s]

AVERTISSEMENT: 2 annotations hors vidéo, exemples: [(201, 1206), (202, 1212)]


001c5339-9a6cdd3e.mov:   0%|          | 0/1205 [00:00<?, ?it/s]

002d290d-f9108a13.mov:   0%|          | 0/1212 [00:00<?, ?it/s]

...

AVERTISSEMENT: 1 annotations hors vidéo, exemples: [(202, 1212)]


02a0f142-1c6504df.mov:   0%|          | 0/1210 [00:00<?, ?it/s]

AVERTISSEMENT: 1 annotations hors vidéo, exemples: [(198, 1188)]


02b12643-f8b3aa6f.mov:   0%|          | 0/1183 [00:00<?, ?it/s]


Total stream frames: 59769
Annotated evaluation frames: 9982
Four-class GT objects: 101011
Four-class GT distribution: {'class1': 75554, 'class0': 16316, 'class3': 6154, 'class2': 2987}


In [8]:
# ============================================================
# 20. SCHEDULER CUE HELPERS
# ============================================================

def scheduler_predictions(
    predictions
):

    return [

        p

        for p
        in predictions

        if float(
            p[4]
        )
        >=
        DECISION_CONF
    ]


def density_score(
    predictions
):

    selected = (
        scheduler_predictions(
            predictions
        )
    )


    return float(

        min(
            len(selected)
            /
            float(
                NMAX_DENSITY
            ),
            1.0
        )
    )


def uncertainty_score(
    predictions
):

    selected = (
        scheduler_predictions(
            predictions
        )
    )


    if len(
        selected
    ) == 0:

        return float(
            U0
        )


    mean_confidence = float(
        np.mean(
            [
                float(p[4])
                for p
                in selected
            ]
        )
    )


    return float(
        1.0
        -
        mean_confidence
    )


# ============================================================
# 21. CONTINUOUS POLICY
# ============================================================

def apply_continuous_policy(
    records
):

    output = []


    for record in records:

        r = (
            record.copy()
        )


        r[
            "policy_preds"
        ] = record[
            "continuous_preds"
        ]


        r[
            "yolo_executed"
        ] = 1


        r[
            "risk_score"
        ] = 1.0


        r[
            "K_t"
        ] = 0


        output.append(
            r
        )


    return output


# ============================================================
# 22. FIXED STRIDE POLICY
# ============================================================

def apply_fixed_stride_policy(
    records,
    stride=3
):

    output = []

    last_predictions = []


    for i, record in enumerate(
        records
    ):

        r = (
            record.copy()
        )


        execute = (
            i % stride
            == 0
        )


        if execute:

            last_predictions = (
                record[
                    "continuous_preds"
                ]
            )

            r[
                "yolo_executed"
            ] = 1

        else:

            r[
                "yolo_executed"
            ] = 0


        r[
            "policy_preds"
        ] = (
            last_predictions
        )


        r[
            "risk_score"
        ] = np.nan


        r[
            "K_t"
        ] = stride


        output.append(
            r
        )


    return output


# ============================================================
# 23. EVENT-DRIVEN POLICY
# ============================================================

def apply_event_policy(
    records,
    theta=
        FIXED_EVENT_THETA,
    max_skip=
        FIXED_EVENT_MAX_SKIP
):

    output = []

    last_predictions = []

    skipped_since_detector = (
        max_skip
    )


    for i, record in enumerate(
        records
    ):

        r = (
            record.copy()
        )


        delta = float(
            record[
                "delta"
            ]
        )


        execute = (

            i == 0

            or

            delta
            >
            theta

            or

            skipped_since_detector
            >=
            max_skip
        )


        if execute:

            last_predictions = (
                record[
                    "continuous_preds"
                ]
            )

            r[
                "yolo_executed"
            ] = 1

            skipped_since_detector = 0


        else:

            r[
                "yolo_executed"
            ] = 0

            skipped_since_detector += 1


        r[
            "policy_preds"
        ] = (
            last_predictions
        )


        r[
            "risk_score"
        ] = delta


        r[
            "K_t"
        ] = max_skip


        output.append(
            r
        )


    return output


# ============================================================
# 24. RAED-VP POLICY
# ============================================================

def apply_raed_policy(
    records,
    delta_ref=
        FIXED_DELTA_REF,
    alpha=
        FIXED_ALPHA,
    beta=
        FIXED_BETA,
    gamma=
        FIXED_GAMMA,
    kmin=
        FIXED_KMIN,
    kmax=
        FIXED_KMAX
):

    output = []

    last_predictions = []

    skipped_since_detector = (
        kmax
    )


    for i, record in enumerate(
        records
    ):

        r = (
            record.copy()
        )


        # ----------------------------------------------------
        # Visual cue
        # ----------------------------------------------------

        delta = float(
            record[
                "delta"
            ]
        )


        delta_norm = float(

            min(
                delta
                /
                (
                    delta_ref
                    +
                    1e-12
                ),
                1.0
            )
        )


        # ----------------------------------------------------
        # Density cue from most recent fresh predictions
        # ----------------------------------------------------

        density = (
            density_score(
                last_predictions
            )
        )


        # ----------------------------------------------------
        # Confidence-derived uncertainty proxy
        # ----------------------------------------------------

        uncertainty = (
            uncertainty_score(
                last_predictions
            )
        )


        # ----------------------------------------------------
        # Risk score
        # ----------------------------------------------------

        risk = (

            alpha
            *
            delta_norm

            +

            beta
            *
            density

            +

            gamma
            *
            uncertainty
        )


        risk = float(
            np.clip(
                risk,
                0.0,
                1.0
            )
        )


        # ----------------------------------------------------
        # Dynamic skip parameter
        # ----------------------------------------------------

        K_t = int(

            kmax

            -

            math.floor(

                (
                    kmax
                    -
                    kmin
                )

                *
                risk
            )
        )


        K_t = int(
            np.clip(
                K_t,
                kmin,
                kmax
            )
        )


        execute = (

            i == 0

            or

            skipped_since_detector
            >=
            K_t
        )


        if execute:

            last_predictions = (
                record[
                    "continuous_preds"
                ]
            )


            r[
                "yolo_executed"
            ] = 1


            skipped_since_detector = 0


        else:

            r[
                "yolo_executed"
            ] = 0


            skipped_since_detector += 1


        # Direct detection reuse
        r[
            "policy_preds"
        ] = (
            last_predictions
        )


        r[
            "risk_score"
        ] = risk


        r[
            "delta_norm"
        ] = delta_norm


        r[
            "density_score"
        ] = density


        r[
            "uncertainty_score"
        ] = uncertainty


        r[
            "K_t"
        ] = K_t


        output.append(
            r
        )


    return output


# ============================================================
# 25. APPLY POLICY SEPARATELY TO EACH VIDEO
# ============================================================

def apply_policy_by_video(
    records,
    policy_function,
    **kwargs
):

    metadata = pd.DataFrame([

        {
            "index":
                i,

            "video":
                record[
                    "video"
                ],

            "frame":
                record[
                    "raw_frame_idx"
                ]
        }

        for i, record
        in enumerate(
            records
        )
    ])


    output = [
        None
    ] * len(
        records
    )


    for video_name, group in (
        metadata.groupby(
            "video"
        )
    ):

        indices = (

            group
            .sort_values(
                "frame"
            )[
                "index"
            ]
            .tolist()
        )


        video_records = [

            records[i]

            for i
            in indices
        ]


        processed = (
            policy_function(
                video_records,
                **kwargs
            )
        )


        for original_index, new_record in zip(
            indices,
            processed
        ):

            output[
                original_index
            ] = (
                new_record
            )


    return output


In [9]:
# ============================================================
# 26. CREATE ALL POLICIES
# ============================================================

continuous_records = (
    apply_policy_by_video(

        all_records,

        apply_continuous_policy
    )
)


stride3_records = (
    apply_policy_by_video(

        all_records,

        apply_fixed_stride_policy,

        stride=
            FIXED_STRIDE
    )
)


event_records = (
    apply_policy_by_video(

        all_records,

        apply_event_policy,

        theta=
            FIXED_EVENT_THETA,

        max_skip=
            FIXED_EVENT_MAX_SKIP
    )
)


raed_records = (
    apply_policy_by_video(

        all_records,

        apply_raed_policy,

        delta_ref=
            FIXED_DELTA_REF,

        alpha=
            FIXED_ALPHA,

        beta=
            FIXED_BETA,

        gamma=
            FIXED_GAMMA,

        kmin=
            FIXED_KMIN,

        kmax=
            FIXED_KMAX
    )
)


# ============================================================
# 27. CLASS-AWARE GREEDY MATCH
# ============================================================

def greedy_match_multiclass(
    predictions,
    ground_truth,
    iou_threshold=0.50,
    confidence_threshold=
        DECISION_CONF
):

    predictions = [

        p

        for p
        in predictions

        if (
            float(
                p[4]
            )
            >=
            confidence_threshold
        )
    ]


    predictions = sorted(

        predictions,

        key=lambda x:
            float(
                x[4]
            ),

        reverse=True
    )


    used_gt = set()

    true_positive = 0


    for prediction in predictions:

        pred_box = (
            prediction[:4]
        )


        pred_class = (
            prediction[5]
        )


        best_iou = 0.0

        best_gt_index = -1


        for j, gt in enumerate(
            ground_truth
        ):

            if j in used_gt:

                continue


            gt_box = (
                gt[:4]
            )


            gt_class = (
                gt[4]
            )


            # SAME CLASS ONLY
            if (
                pred_class
                !=
                gt_class
            ):

                continue


            current_iou = (
                box_iou(
                    pred_box,
                    gt_box
                )
            )


            if (
                current_iou
                >
                best_iou
            ):

                best_iou = (
                    current_iou
                )

                best_gt_index = j


        if (
            best_iou
            >=
            iou_threshold

            and

            best_gt_index
            >= 0
        ):

            true_positive += 1

            used_gt.add(
                best_gt_index
            )


    false_positive = (
        len(
            predictions
        )
        -
        true_positive
    )


    false_negative = (
        len(
            ground_truth
        )
        -
        true_positive
    )


    return (
        true_positive,
        false_positive,
        false_negative
    )


In [10]:
# ============================================================
# 28. MICRO PRECISION / RECALL / F1
# ============================================================

def evaluate_detection_metrics(
    records,
    iou_threshold=0.50
):

    total_tp = 0
    total_fp = 0
    total_fn = 0

    evaluated_frames = 0

    count_errors = []


    for record in records:

        ground_truth = (
            record[
                "gt_objects"
            ]
        )


        # Ignore raw frames without a 5-FPS
        # annotation timestamp.
        if ground_truth is None:

            continue


        evaluated_frames += 1


        predictions = (
            record[
                "policy_preds"
            ]
        )


        (
            tp,
            fp,
            fn
        ) = greedy_match_multiclass(

            predictions,

            ground_truth,

            iou_threshold=
                iou_threshold,

            confidence_threshold=
                DECISION_CONF
        )


        total_tp += tp
        total_fp += fp
        total_fn += fn


        filtered_predictions = [

            p

            for p
            in predictions

            if (
                float(
                    p[4]
                )
                >=
                DECISION_CONF
            )
        ]


        count_errors.append(

            abs(
                len(
                    filtered_predictions
                )
                -
                len(
                    ground_truth
                )
            )
        )


    precision = (

        total_tp

        /
        (
            total_tp
            +
            total_fp
            +
            1e-12
        )
    )


    recall = (

        total_tp

        /
        (
            total_tp
            +
            total_fn
            +
            1e-12
        )
    )


    f1 = (

        2.0
        *
        precision
        *
        recall

        /
        (
            precision
            +
            recall
            +
            1e-12
        )
    )


    return {

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                f1
            ),

        "count_mae":
            float(
                np.mean(
                    count_errors
                )
            )
            if count_errors
            else np.nan,

        "tp":
            int(
                total_tp
            ),

        "fp":
            int(
                total_fp
            ),

        "fn":
            int(
                total_fn
            ),

        "evaluated_frames":
            int(
                evaluated_frames
            )
    }


# ============================================================
# 29. AP INTEGRATION
# ============================================================

def compute_ap_from_pr(
    recalls,
    precisions
):

    recalls = np.asarray(
        recalls,
        dtype=float
    )


    precisions = np.asarray(
        precisions,
        dtype=float
    )


    mrec = np.concatenate(
        (
            [0.0],
            recalls,
            [1.0]
        )
    )


    mpre = np.concatenate(
        (
            [0.0],
            precisions,
            [0.0]
        )
    )


    # Precision envelope
    for i in range(
        len(mpre) - 2,
        -1,
        -1
    ):

        mpre[i] = max(
            mpre[i],
            mpre[i + 1]
        )


    change_indices = np.where(

        mrec[1:]
        !=
        mrec[:-1]

    )[0]


    ap = np.sum(

        (
            mrec[
                change_indices + 1
            ]
            -
            mrec[
                change_indices
            ]
        )

        *

        mpre[
            change_indices + 1
        ]
    )


    return float(ap)


# ============================================================
# 30. AP FOR ONE CLASS
# ============================================================

def evaluate_ap_for_class(
    records,
    class_name,
    iou_threshold=0.50
):

    evaluated_records = [

        record

        for record
        in records

        if (
            record[
                "gt_objects"
            ]
            is not None
        )
    ]


    all_predictions = []

    total_gt = 0


    # --------------------------------------------------------
    # Gather all predictions and GT for this class
    # --------------------------------------------------------

    for frame_index, record in enumerate(
        evaluated_records
    ):

        class_gt = [

            gt

            for gt
            in record[
                "gt_objects"
            ]

            if (
                gt[4]
                ==
                class_name
            )
        ]


        class_predictions = [

            pred

            for pred
            in record[
                "policy_preds"
            ]

            if (
                pred[5]
                ==
                class_name
            )
        ]


        total_gt += len(
            class_gt
        )


        for prediction in (
            class_predictions
        ):

            all_predictions.append({

                "frame_index":
                    frame_index,

                "box":
                    prediction[:4],

                "confidence":
                    float(
                        prediction[4]
                    )
            })


    if total_gt == 0:

        return np.nan


    # --------------------------------------------------------
    # Global confidence sorting
    # --------------------------------------------------------

    all_predictions = sorted(

        all_predictions,

        key=lambda x:
            x[
                "confidence"
            ],

        reverse=True
    )


    # --------------------------------------------------------
    # Used GT by frame
    # --------------------------------------------------------

    used_gt = {

        frame_index:
            set()

        for frame_index
        in range(
            len(
                evaluated_records
            )
        )
    }


    tp = np.zeros(
        len(
            all_predictions
        ),
        dtype=float
    )


    fp = np.zeros(
        len(
            all_predictions
        ),
        dtype=float
    )


    # --------------------------------------------------------
    # Match each prediction
    # --------------------------------------------------------

    for pred_index, prediction in enumerate(
        all_predictions
    ):

        frame_index = (
            prediction[
                "frame_index"
            ]
        )


        class_gt = [

            gt

            for gt
            in evaluated_records[
                frame_index
            ][
                "gt_objects"
            ]

            if (
                gt[4]
                ==
                class_name
            )
        ]


        best_iou = 0.0

        best_gt_index = -1


        for gt_index, gt in enumerate(
            class_gt
        ):

            if (
                gt_index
                in
                used_gt[
                    frame_index
                ]
            ):

                continue


            current_iou = (
                box_iou(

                    prediction[
                        "box"
                    ],

                    gt[:4]
                )
            )


            if (
                current_iou
                >
                best_iou
            ):

                best_iou = (
                    current_iou
                )

                best_gt_index = (
                    gt_index
                )


        if (
            best_iou
            >=
            iou_threshold

            and

            best_gt_index
            >= 0
        ):

            tp[
                pred_index
            ] = 1.0


            used_gt[
                frame_index
            ].add(
                best_gt_index
            )


        else:

            fp[
                pred_index
            ] = 1.0


    cumulative_tp = (
        np.cumsum(
            tp
        )
    )


    cumulative_fp = (
        np.cumsum(
            fp
        )
    )


    recalls = (

        cumulative_tp

        /
        (
            total_gt
            +
            1e-12
        )
    )


    precisions = (

        cumulative_tp

        /
        (
            cumulative_tp
            +
            cumulative_fp
            +
            1e-12
        )
    )


    return (
        compute_ap_from_pr(
            recalls,
            precisions
        )
    )


# ============================================================
# 31. TRUE MULTI-CLASS mAP
# ============================================================

def evaluate_multiclass_map(
    records
):

    iou_thresholds = (
        np.arange(
            0.50,
            0.96,
            0.05
        )
    )


    per_class_rows = []


    for class_name in (
        EVAL_CLASSES
    ):

        ap50 = (
            evaluate_ap_for_class(

                records,

                class_name,

                iou_threshold=
                    0.50
            )
        )


        aps = []


        for threshold in (
            iou_thresholds
        ):

            ap = (
                evaluate_ap_for_class(

                    records,

                    class_name,

                    iou_threshold=
                        float(
                            threshold
                        )
                )
            )


            aps.append(
                ap
            )


        valid_aps = [

            x

            for x
            in aps

            if not np.isnan(x)
        ]


        ap5095 = (

            float(
                np.mean(
                    valid_aps
                )
            )

            if valid_aps

            else np.nan
        )


        per_class_rows.append({

            "Class":
                class_name,

            "AP@0.5":
                ap50,

            "AP@0.5:0.95":
                ap5095
        })


    per_class_df = (
        pd.DataFrame(
            per_class_rows
        )
    )


    macro_map50 = float(

        per_class_df[
            "AP@0.5"
        ]
        .dropna()
        .mean()
    )


    macro_map5095 = float(

        per_class_df[
            "AP@0.5:0.95"
        ]
        .dropna()
        .mean()
    )


    return {

        "mAP@0.5":
            macro_map50,

        "mAP@0.5:0.95":
            macro_map5095,

        "per_class":
            per_class_df
    }


# ============================================================
# 32. PER-CLASS PRECISION / RECALL / F1
# ============================================================

def evaluate_per_class_pr(
    records,
    iou_threshold=0.50
):

    rows = []


    for class_name in (
        EVAL_CLASSES
    ):

        total_tp = 0
        total_fp = 0
        total_fn = 0


        for record in records:

            if (
                record[
                    "gt_objects"
                ]
                is None
            ):

                continue


            gt = [

                x

                for x
                in record[
                    "gt_objects"
                ]

                if (
                    x[4]
                    ==
                    class_name
                )
            ]


            preds = [

                x

                for x
                in record[
                    "policy_preds"
                ]

                if (
                    x[5]
                    ==
                    class_name

                    and

                    float(
                        x[4]
                    )
                    >=
                    DECISION_CONF
                )
            ]


            (
                tp,
                fp,
                fn
            ) = greedy_match_multiclass(

                preds,

                gt,

                iou_threshold=
                    iou_threshold,

                confidence_threshold=
                    DECISION_CONF
            )


            total_tp += tp
            total_fp += fp
            total_fn += fn


        precision = (

            total_tp

            /
            (
                total_tp
                +
                total_fp
                +
                1e-12
            )
        )


        recall = (

            total_tp

            /
            (
                total_tp
                +
                total_fn
                +
                1e-12
            )
        )


        f1 = (

            2
            *
            precision
            *
            recall

            /
            (
                precision
                +
                recall
                +
                1e-12
            )
        )


        rows.append({

            "Class":
                class_name,

            "Precision":
                precision,

            "Recall":
                recall,

            "F1":
                f1,

            "TP":
                total_tp,

            "FP":
                total_fp,

            "FN":
                total_fn
        })


    return pd.DataFrame(
        rows
    )


In [11]:
# ============================================================
# 33. POLICY SUMMARY
# ============================================================

def summarize_policy(
    records,
    policy_name
):

    n_stream_frames = len(
        records
    )


    n_yolo_frames = int(
        sum(
            int(
                record[
                    "yolo_executed"
                ]
            )
            for record
            in records
        )
    )


    n_skipped = (
        n_stream_frames
        -
        n_yolo_frames
    )


    frame_reduction = (

        n_skipped

        /
        (
            n_stream_frames
            +
            1e-12
        )
    )


    # --------------------------------------------------------
    # Detection metrics
    # --------------------------------------------------------

    detection_metrics = (
        evaluate_detection_metrics(
            records
        )
    )


    map_metrics = (
        evaluate_multiclass_map(
            records
        )
    )


    # --------------------------------------------------------
    # Detector-time proxy
    #
    # We have actually measured detector latency for each
    # raw frame during continuous cache generation.
    #
    # For an adaptive policy, skipped frames contribute
    # zero detector time.
    #
    # Scheduler overhead is NOT included here.
    # --------------------------------------------------------

    total_detector_ms = 0.0


    for record in records:

        if (
            record[
                "yolo_executed"
            ]
            == 1
        ):

            total_detector_ms += float(
                record[
                    "continuous_latency_ms"
                ]
            )


    detector_time_proxy_ms = (

        total_detector_ms

        /
        (
            n_stream_frames
            +
            1e-12
        )
    )


    detector_rate_proxy = (

        1000.0

        /
        (
            detector_time_proxy_ms
            +
            1e-12
        )
    )


    return {

        "Policy":
            policy_name,

        "Stream Frames":
            n_stream_frames,

        "Evaluated Frames":
            detection_metrics[
                "evaluated_frames"
            ],

        "YOLO Frames":
            n_yolo_frames,

        "Skipped Frames":
            n_skipped,

        "Frame Reduction (%)":
            frame_reduction
            *
            100.0,

        "mAP@0.5 (%)":
            map_metrics[
                "mAP@0.5"
            ]
            *
            100.0,

        "mAP@0.5:0.95 (%)":
            map_metrics[
                "mAP@0.5:0.95"
            ]
            *
            100.0,

        "Precision (%)":
            detection_metrics[
                "precision"
            ]
            *
            100.0,

        "Recall (%)":
            detection_metrics[
                "recall"
            ]
            *
            100.0,

        "F1 (%)":
            detection_metrics[
                "f1"
            ]
            *
            100.0,

        "Count MAE":
            detection_metrics[
                "count_mae"
            ],

        "TP":
            detection_metrics[
                "tp"
            ],

        "FP":
            detection_metrics[
                "fp"
            ],

        "FN":
            detection_metrics[
                "fn"
            ],

        "Detector-time Proxy (ms/frame)":
            detector_time_proxy_ms,

        "Detector-rate Proxy (FPS)":
            detector_rate_proxy
    }


In [16]:
# ============================================================
# 34. COMPUTE MAIN RESULTS
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "COMPUTING MULTI-CLASS RESULTS"
)

print(
    "=" * 100
)


summary_rows = []


for policy_name, records in [

    (
        "Continuous",
        continuous_records
    ),

    (
        "Fixed Stride-3",
        stride3_records
    ),

    (
        "Event-driven",
        event_records
    ),

    (
        "RAED-VP",
        raed_records
    )
]:

    print(
        "\nEvaluating:",
        policy_name
    )


    result = (
        summarize_policy(

            records,

            policy_name
        )
    )


    summary_rows.append(
        result
    )


summary_df = pd.DataFrame(
    summary_rows
)


pd.set_option(
    "display.max_columns",
    None
)


print(
    "\n" + "=" * 100
)

print(
    "FINAL FOUR-CLASS SUMMARY"
)

print(
    "=" * 100
)


display(
    summary_df.round(3)
)



COMPUTING MULTI-CLASS RESULTS

Evaluating: Continuous

Evaluating: Fixed Stride-3

Evaluating: Event-driven

Evaluating: RAED-VP

FINAL FOUR-CLASS SUMMARY


,Policy,Stream Frames,Evaluated Frames,YOLO Frames,Skipped Frames,Frame Reduction (%),mAP@0.5 (%),mAP@0.5:0.95 (%),Precision (%),Recall (%),F1 (%),Count MAE,TP,FP,FN,Detector-time Proxy (ms/frame),Detector-rate Proxy (FPS)
0,Continuous,59769,9982,59769,0,0.000,40.782,21.520,80.506,48.167,60.273,4.225,48654,11781,52357,12.930,77.342
1,Fixed Stride-3,59769,9982,19936,39833,66.645,40.782,21.520,80.506,48.167,60.273,4.225,48654,11781,52357,4.377,228.447
2,Event-driven,59769,9982,39960,19809,33.143,40.834,21.562,80.558,48.325,60.411,4.206,48814,11781,52197,8.710,114.814
3,RAED-VP,59769,9982,18211,41558,69.531,40.807,21.375,80.515,48.101,60.223,4.225,48587,11758,52424,3.997,250.201


In [17]:
# ============================================================
# 35. PER-CLASS AP TABLES
# ============================================================

all_per_class_ap = []


for policy_name, records in [

    (
        "Continuous",
        continuous_records
    ),

    (
        "Fixed Stride-3",
        stride3_records
    ),

    (
        "Event-driven",
        event_records
    ),

    (
        "RAED-VP",
        raed_records
    )
]:

    map_result = (
        evaluate_multiclass_map(
            records
        )
    )


    temp = (
        map_result[
            "per_class"
        ]
        .copy()
    )


    temp.insert(
        0,
        "Policy",
        policy_name
    )


    temp[
        "AP@0.5"
    ] *= 100.0


    temp[
        "AP@0.5:0.95"
    ] *= 100.0


    all_per_class_ap.append(
        temp
    )


per_class_ap_df = (
    pd.concat(
        all_per_class_ap,
        ignore_index=True
    )
)


print(
    "\n" + "=" * 100
)

print(
    "PER-CLASS AP RESULTS"
)

print(
    "=" * 100
)


display(
    per_class_ap_df.round(2)
)


# ============================================================
# 36. PER-CLASS PRECISION / RECALL / F1
# ============================================================

all_per_class_pr = []


for policy_name, records in [

    (
        "Continuous",
        continuous_records
    ),

    (
        "Fixed Stride-3",
        stride3_records
    ),

    (
        "Event-driven",
        event_records
    ),

    (
        "RAED-VP",
        raed_records
    )
]:

    temp = (
        evaluate_per_class_pr(
            records
        )
    )


    temp.insert(
        0,
        "Policy",
        policy_name
    )


    temp[
        "Precision"
    ] *= 100.0


    temp[
        "Recall"
    ] *= 100.0


    temp[
        "F1"
    ] *= 100.0


    all_per_class_pr.append(
        temp
    )


per_class_pr_df = (
    pd.concat(
        all_per_class_pr,
        ignore_index=True
    )
)


print(
    "\n" + "=" * 100
)

print(
    "PER-CLASS PRECISION / RECALL / F1"
)

print(
    "=" * 100
)


display(
    per_class_pr_df.round(2)
)



PER-CLASS AP RESULTS


,Policy,Class,AP@0.5,AP@0.5:0.95
0,Continuous,class0,35.75,13.50
1,Continuous,class1,66.99,34.90
2,Continuous,class2,35.38,24.43
3,Continuous,class3,25.01,13.25
4,Fixed Stride-3,class0,35.75,13.50
5,Fixed Stride-3,class1,66.99,34.90
6,Fixed Stride-3,class2,35.38,24.43
7,Fixed Stride-3,class3,25.01,13.25
8,Event-driven,class0,35.81,13.49
9,Event-driven,class1,67.13,35.03



PER-CLASS PRECISION / RECALL / F1


,Policy,Class,Precision,Recall,F1,TP,FP,FN
0,Continuous,class0,75.41,20.10,31.74,3279,1069,13037
1,Continuous,class1,82.07,56.74,67.09,42866,9367,32688
2,Continuous,class2,75.25,33.28,46.15,994,327,1993
3,Continuous,class3,59.81,24.62,34.88,1515,1018,4639
4,Fixed Stride-3,class0,75.41,20.10,31.74,3279,1069,13037
5,Fixed Stride-3,class1,82.07,56.74,67.09,42866,9367,32688
6,Fixed Stride-3,class2,75.25,33.28,46.15,994,327,1993
7,Fixed Stride-3,class3,59.81,24.62,34.88,1515,1018,4639
8,Event-driven,class0,75.27,20.23,31.88,3300,1084,13016
9,Event-driven,class1,82.14,56.93,67.25,43014,9351,32540


### RAED-VP + Motion Compensation

In [18]:
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import cv2

from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm
from IPython.display import display


# ================================================================
# 1. CONFIGURATION
# ================================================================

# Seuil de matching à fixer avant l'évaluation.
# Utiliser celui de UA-DETRAC si vous l'avez déjà défini.
MC_MATCH_IOU = 0.10

required_variables = [
    "raed_records",
    "valid_videos",
    "summarize_policy",
    "evaluate_multiclass_map",
    "evaluate_per_class_pr",
    "DECISION_CONF",
    "EVAL_CLASSES",
    "OUTPUT_DIR",
    "summary_df"
]

missing = [
    name for name in required_variables
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Exécutez d'abord toute l'évaluation principale. "
        f"Variables manquantes : {missing}"
    )

assert len(raed_records) > 0, "raed_records est vide."

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("RAED-VP + MOTION COMPENSATION — BDD100K")
print("=" * 65)

print("Frames disponibles :", len(raed_records))
print("Vidéos :", len(valid_videos))
print("Seuil IoU Hungarian :", MC_MATCH_IOU)


# ================================================================
# 2. DIMENSIONS DES VIDÉOS
# ================================================================

video_dims = {}

for row in valid_videos.itertuples(index=False):

    cap = cv2.VideoCapture(str(row.video_path))

    if not cap.isOpened():
        raise RuntimeError(
            f"Vidéo illisible : {row.video_path}"
        )

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    cap.release()

    assert width > 0 and height > 0

    video_name = Path(row.video_path).stem

    video_dims[video_name] = (width, height)


# ================================================================
# 3. IoU ENTRE DEUX BOUNDING BOXES
# ================================================================

def mc_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = (
        max(0.0, x2 - x1)
        * max(0.0, y2 - y1)
    )

    area1 = (
        max(0.0, box1[2] - box1[0])
        * max(0.0, box1[3] - box1[1])
    )

    area2 = (
        max(0.0, box2[2] - box2[0])
        * max(0.0, box2[3] - box2[1])
    )

    union = area1 + area2 - intersection

    return intersection / max(union, 1e-12)


# ================================================================
# 4. HUNGARIAN MATCHING + ESTIMATION DU MOUVEMENT
# ================================================================

def mc_estimate_velocity(previous, current, gap):

    # Une vitesse (vx, vy) pour chaque nouvelle détection
    velocities = np.zeros(
        (len(current), 2),
        dtype=np.float64
    )

    if not previous or not current or gap <= 0:
        return velocities, 0

    n_pairs = 0

    # Association séparée pour chaque classe
    for cls in EVAL_CLASSES:

        old_idx = [
            i for i, p in enumerate(previous)
            if p[5] == cls
            and p[4] >= DECISION_CONF
        ]

        new_idx = [
            j for j, p in enumerate(current)
            if p[5] == cls
            and p[4] >= DECISION_CONF
        ]

        if not old_idx or not new_idx:
            continue

        similarities = np.array([
            [
                mc_iou(previous[i], current[j])
                for j in new_idx
            ]
            for i in old_idx
        ], dtype=float)

        # Hungarian assignment
        rows, cols = linear_sum_assignment(
            1.0 - similarities
        )

        for i, j in zip(rows, cols):

            if similarities[i, j] < MC_MATCH_IOU:
                continue

            old = previous[old_idx[i]]
            new = current[new_idx[j]]

            # Centres des anciennes et nouvelles boîtes
            old_cx = (old[0] + old[2]) / 2.0
            old_cy = (old[1] + old[3]) / 2.0

            new_cx = (new[0] + new[2]) / 2.0
            new_cy = (new[1] + new[3]) / 2.0

            # Vitesse en pixels par frame vidéo
            vx = (new_cx - old_cx) / gap
            vy = (new_cy - old_cy) / gap

            velocities[new_idx[j]] = [vx, vy]

            n_pairs += 1

    return velocities, n_pairs


# ================================================================
# 5. PROPAGATION DES BOUNDING BOXES
# ================================================================

def mc_propagate(anchor, velocities, age, width, height):

    compensated = []

    for j, p in enumerate(anchor):

        vx, vy = velocities[j]

        dx = vx * age
        dy = vy * age

        # Translating the bounding box
        x1 = float(np.clip(p[0] + dx, 0, width))
        y1 = float(np.clip(p[1] + dy, 0, height))

        x2 = float(np.clip(p[2] + dx, 0, width))
        y2 = float(np.clip(p[3] + dy, 0, height))

        if x2 <= x1 or y2 <= y1:
            continue

        # Classe et confiance conservées
        compensated.append((
            x1,
            y1,
            x2,
            y2,
            p[4],
            p[5]
        ))

    return compensated


# ================================================================
# 6. APPLIQUER MOTION COMPENSATION À RAED-VP
# ================================================================

mc_records = []

current_video = None

last_fresh = []
last_detector_frame = None

current_velocity = np.empty(
    (0, 2),
    dtype=float
)

n_matches = 0

matching_ms = 0.0
propagation_ms = 0.0

per_video = {}
seen_videos = set()


for record in tqdm(
    raed_records,
    desc="RAED-VP Motion Compensation"
):

    video = record["video"]
    frame_idx = int(record["raw_frame_idx"])

    # --------------------------------------------
    # Réinitialisation à chaque nouvelle vidéo
    # --------------------------------------------

    if video != current_video:

        if video in seen_videos:
            raise RuntimeError(
                "Les frames vidéo ne sont pas contiguës."
            )

        seen_videos.add(video)

        current_video = video

        last_fresh = []
        last_detector_frame = None

        current_velocity = np.empty(
            (0, 2),
            dtype=float
        )

    if video not in video_dims:
        raise RuntimeError(
            f"Dimensions introuvables pour {video}"
        )

    width, height = video_dims[video]

    # Même décision que RAED-VP Direct Reuse
    execute = int(record["yolo_executed"]) == 1

    # --------------------------------------------
    # A. FRAME AVEC YOLO EXÉCUTÉ
    # --------------------------------------------

    if execute:

        fresh = record["continuous_preds"]

        if last_detector_frame is None:
            gap = 0
        else:
            gap = frame_idx - last_detector_frame

        t0 = time.perf_counter()

        current_velocity, n = mc_estimate_velocity(
            last_fresh,
            fresh,
            gap
        )

        elapsed = (
            time.perf_counter() - t0
        ) * 1000.0

        matching_ms += elapsed
        n_matches += n

        # Mettre à jour l'ancre de détection
        last_fresh = fresh
        last_detector_frame = frame_idx

        # Sur une frame fraîche, aucune propagation
        compensated = fresh

    # --------------------------------------------
    # B. FRAME SAUTÉE : COMPENSATION
    # --------------------------------------------

    else:

        if last_detector_frame is None:
            raise RuntimeError(
                f"Frame sautée sans YOLO initial : {video}"
            )

        age = frame_idx - last_detector_frame

        t0 = time.perf_counter()

        compensated = mc_propagate(
            last_fresh,
            current_velocity,
            age,
            width,
            height
        )

        elapsed = (
            time.perf_counter() - t0
        ) * 1000.0

        propagation_ms += elapsed
        n = 0

    # --------------------------------------------
    # Construire le nouvel enregistrement
    # --------------------------------------------

    modified = record.copy()

    # Seules les bounding boxes changent
    modified["policy_preds"] = compensated

    mc_records.append(modified)

    # --------------------------------------------
    # Statistiques par vidéo
    # --------------------------------------------

    if video not in per_video:

        per_video[video] = {
            "videoName": video,
            "stream_frames": 0,
            "yolo_frames": 0,
            "annotated_frames": 0,
            "reused_annotated": 0,
            "matches": 0,
            "matching_ms": 0.0,
            "propagation_ms": 0.0
        }

    v = per_video[video]

    is_labeled = record["gt_objects"] is not None

    v["stream_frames"] += 1
    v["yolo_frames"] += int(execute)
    v["annotated_frames"] += int(is_labeled)

    v["reused_annotated"] += int(
        is_labeled and not execute
    )

    v["matches"] += n

    if execute:
        v["matching_ms"] += elapsed
    else:
        v["propagation_ms"] += elapsed


# ================================================================
# 7. VÉRIFIER LES DÉCISIONS YOLO
# ================================================================

assert len(mc_records) == len(raed_records)

assert (
    [r["yolo_executed"] for r in mc_records]
    ==
    [r["yolo_executed"] for r in raed_records]
), "Les décisions YOLO ont changé !"

assert len(seen_videos) == len(valid_videos)

n_frames = len(raed_records)

n_yolo = sum(
    int(r["yolo_executed"])
    for r in raed_records
)

print("\n===== VÉRIFICATION =====")

print("Frames totales :", n_frames)
print("Exécutions YOLO :", n_yolo)
print("Vidéos :", len(seen_videos))
print("Appariements Hungarian :", n_matches)

print(
    "Frame Reduction (%) :",
    round(100 * (1 - n_yolo / n_frames), 3)
)


# ================================================================
# 8. MÉTRIQUES : MÊMES FONCTIONS QUE DIRECT REUSE
# ================================================================

mc_summary = summarize_policy(
    mc_records,
    "RAED-VP + Motion Compensation"
)

direct_rows = summary_df.loc[
    summary_df["Policy"] == "RAED-VP"
]

assert len(direct_rows) == 1, (
    "RAED-VP absent de summary_df"
)

direct_summary = direct_rows.iloc[0].to_dict()

assert (
    int(direct_summary["YOLO Frames"])
    ==
    int(mc_summary["YOLO Frames"])
)


# ================================================================
# 9. TEMPS D'EXÉCUTION ET SURCOÛT
# ================================================================

mc_extra_ms = matching_ms + propagation_ms

# Temps YOLO déjà mesuré lors de l'évaluation initiale
base_detector_ms = sum(
    float(r["continuous_latency_ms"])
    for r in raed_records
    if r["yolo_executed"] == 1
)

comparison = []

for name, row, extra_ms in [

    (
        "RAED-VP (Direct Reuse)",
        direct_summary,
        0.0
    ),

    (
        "RAED-VP + Motion Compensation",
        mc_summary,
        mc_extra_ms
    )
]:

    # Proxy : détecteur + opérations de compensation
    total_proxy_ms = base_detector_ms + extra_ms

    avg_ms = total_proxy_ms / n_frames

    comparison.append({

        "Policy": name,

        "Stream Frames": n_frames,

        "Annotated Frames": int(
            row["Evaluated Frames"]
        ),

        "YOLO Frames": int(
            row["YOLO Frames"]
        ),

        "Frame Reduction (%)": float(
            row["Frame Reduction (%)"]
        ),

        "mAP@0.5 (%)": float(
            row["mAP@0.5 (%)"]
        ),

        "mAP@0.5:0.95 (%)": float(
            row["mAP@0.5:0.95 (%)"]
        ),

        "Precision (%)": float(
            row["Precision (%)"]
        ),

        "Recall (%)": float(
            row["Recall (%)"]
        ),

        "F1 (%)": float(
            row["F1 (%)"]
        ),

        "Detector + MC proxy (ms/frame)": avg_ms,

        "Detector + MC proxy (FPS)": (
            1000.0 / avg_ms
            if avg_ms > 0 else np.nan
        ),

        "Motion overhead (ms/frame)": (
            extra_ms / n_frames
        )
    })


# ================================================================
# 10. TABLEAUX
# ================================================================

mc_compare_df = pd.DataFrame(comparison)

mc_per_class_ap = evaluate_multiclass_map(
    mc_records
)["per_class"]

mc_per_class_pr = evaluate_per_class_pr(
    mc_records
)

mc_per_video_df = pd.DataFrame(
    per_video.values()
)

pd.set_option("display.max_columns", None)

print("\n============================================")
print("DIRECT REUSE VS MOTION COMPENSATION")
print("============================================")

display(mc_compare_df.round(3))

print("\n===== AP PAR CLASSE =====")

display(mc_per_class_ap.round(3))

print("\n===== PRECISION / RECALL / F1 PAR CLASSE =====")

display(mc_per_class_pr.round(3))


# ================================================================
# 11. SAUVEGARDER LES RÉSULTATS
# ================================================================

mc_compare_df.to_csv(
    OUTPUT_DIR / "motion_compensation_comparison.csv",
    index=False
)

mc_per_class_ap.to_csv(
    OUTPUT_DIR / "motion_compensation_AP_per_class.csv",
    index=False
)

mc_per_class_pr.to_csv(
    OUTPUT_DIR / "motion_compensation_PRF1_per_class.csv",
    index=False
)

mc_per_video_df.to_csv(
    OUTPUT_DIR / "motion_compensation_runtime_by_video.csv",
    index=False
)

mc_config = {

    "method": (
        "Class-aware Hungarian IoU matching "
        "+ constant-velocity bounding-box translation"
    ),

    "match_iou_min": MC_MATCH_IOU,

    "association_confidence": DECISION_CONF,

    "videos": len(seen_videos),

    "timing_note": (
        "Detector + motion compensation proxy. "
        "Video decoding, scheduler overhead and "
        "other pipeline costs are excluded."
    )
}

(
    OUTPUT_DIR / "motion_compensation_config.json"
).write_text(
    json.dumps(mc_config, indent=2),
    encoding="utf-8"
)


# ================================================================
# 12. RAPPORT FINAL
# ================================================================

print("\n========== SURCOÛT MOTION COMPENSATION ==========")

print(
    "Matching Hungarian (secondes) :",
    round(matching_ms / 1000, 3)
)

print(
    "Propagation (secondes) :",
    round(propagation_ms / 1000, 3)
)

print(
    "Surcoût moyen (ms/frame) :",
    round(mc_extra_ms / n_frames, 4)
)

reused_annotated = sum(
    r["yolo_executed"] == 0
    and r["gt_objects"] is not None
    for r in raed_records
)

n_annotated = sum(
    r["gt_objects"] is not None
    for r in raed_records
)

print(
    "Frames annotées avec réutilisation :",
    reused_annotated,
    "/",
    n_annotated
)

print(
    "Pourcentage réutilisé (%) :",
    round(
        100 * reused_annotated / n_annotated,
        3
    )
)

print("\n✅ Motion Compensation terminée")

print("Résultats sauvegardés dans :", OUTPUT_DIR)

print(
    "\nNote : les FPS sont des estimations "
    "détecteur + compensation, pas des "
    "mesures end-to-end indépendantes."
)


RAED-VP + MOTION COMPENSATION — BDD100K
Frames disponibles : 59769
Vidéos : 50
Seuil IoU Hungarian : 0.1


RAED-VP Motion Compensation:   0%|          | 0/59769 [00:00<?, ?it/s]


===== VÉRIFICATION =====
Frames totales : 59769
Exécutions YOLO : 18211
Vidéos : 50
Appariements Hungarian : 100212
Frame Reduction (%) : 69.531

DIRECT REUSE VS MOTION COMPENSATION


,Policy,Stream Frames,Annotated Frames,YOLO Frames,Frame Reduction (%),mAP@0.5 (%),mAP@0.5:0.95 (%),Precision (%),Recall (%),F1 (%),Detector + MC proxy (ms/frame),Detector + MC proxy (FPS),Motion overhead (ms/frame)
0,RAED-VP (Direct Reuse),59769,9982,18211,69.531,40.807,21.375,80.515,48.101,60.223,3.997,250.201,0.000
1,RAED-VP + Motion Compensation,59769,9982,18211,69.531,40.680,21.188,80.142,47.877,59.944,6.673,149.861,2.676



===== AP PAR CLASSE =====


,Class,AP@0.5,AP@0.5:0.95
0,class0,0.356,0.133
1,class1,0.667,0.343
2,class2,0.353,0.239
3,class3,0.252,0.132



===== PRECISION / RECALL / F1 PAR CLASSE =====


,Class,Precision,Recall,F1,TP,FP,FN
0,class0,0.742,0.198,0.312,3225,1124,13091
1,class1,0.817,0.565,0.668,42651,9529,32903
2,class2,0.746,0.328,0.455,979,333,2008
3,class3,0.602,0.245,0.348,1506,997,4648



========== SURCOÛT MOTION COMPENSATION ==========
Matching Hungarian (secondes) : 4.202
Propagation (secondes) : 155.743
Surcoût moyen (ms/frame) : 2.6761
Frames annotées avec réutilisation : 5169 / 9982
Pourcentage réutilisé (%) : 51.783

✅ Motion Compensation terminée
Résultats sauvegardés dans : /kaggle/working/raed_bdd_multiclass_validation

Note : les FPS sont des estimations détecteur + compensation, pas des mesures end-to-end indépendantes.


In [19]:

# ============================================================
# DIAGNOSTIC : DIRECT REUSE VS MOTION COMPENSATION
# SUR LES FRAMES ANNOTÉES DE BDD100K
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

assert len(raed_records) == len(mc_records)

pairs = list(zip(raed_records, mc_records))

# Vérifier la correspondance exacte des frames
for direct, motion in pairs:
    assert direct["video"] == motion["video"]
    assert direct["raw_frame_idx"] == motion["raw_frame_idx"]
    assert direct["yolo_executed"] == motion["yolo_executed"]

# Conserver les frames avec GT
annotated = [
    (d, m) for d, m in pairs
    if d["gt_objects"] is not None
]

# Frames annotées avec YOLO exécuté
fresh = [
    (d, m) for d, m in annotated
    if int(d["yolo_executed"]) == 1
]

# Frames annotées avec détections réutilisées
reused = [
    (d, m) for d, m in annotated
    if int(d["yolo_executed"]) == 0
]

print("========== DISTRIBUTION ==========")

print("Frames annotées :", len(annotated))
print("Frames fraîches :", len(fresh))
print("Frames réutilisées :", len(reused))

print(
    "Réutilisation annotée (%) :",
    round(100 * len(reused) / len(annotated), 3)
    if annotated else np.nan
)

# Sur les frames fraîches, les prédictions
# doivent être strictement identiques.
fresh_equal = all(
    d["policy_preds"] == m["policy_preds"]
    for d, m in fresh
)

print("Prédictions fraîches identiques :", fresh_equal)

assert fresh_equal, (
    "ERREUR : Motion Compensation modifie "
    "les prédictions des frames fraîches."
)

assert reused, "Aucune frame annotée avec réutilisation."

# ============================================================
# ÉVALUATION UNIQUEMENT DES FRAMES RÉUTILISÉES
# ============================================================

diagnostic_rows = []

for name, index in [
    ("Direct Reuse", 0),
    ("Motion Compensation", 1)
]:

    records = [
        pair[index] for pair in reused
    ]

    det50 = evaluate_detection_metrics(
        records,
        iou_threshold=0.50
    )

    det75 = evaluate_detection_metrics(
        records,
        iou_threshold=0.75
    )

    ap = evaluate_multiclass_map(records)

    diagnostic_rows.append({
        "Méthode": name,
        "Frames GT": len(records),
        "mAP50 (%)": 100 * ap["mAP@0.5"],
        "mAP50-95 (%)": 100 * ap["mAP@0.5:0.95"],
        "Precision@50 (%)": 100 * det50["precision"],
        "Recall@50 (%)": 100 * det50["recall"],
        "Recall@75 (%)": 100 * det75["recall"],
        "TP@50": det50["tp"],
        "FP@50": det50["fp"],
        "FN@50": det50["fn"]
    })

diagnostic_df = pd.DataFrame(diagnostic_rows)

print("\n========== FRAMES RÉUTILISÉES UNIQUEMENT ==========")
display(diagnostic_df.round(3))

# ============================================================
# VÉRIFIER LE DÉPLACEMENT DES BOXES
# ============================================================

total_before = 0
total_after = 0
moved = 0
same = 0
displacements = []

for direct, motion in reused:

    before = direct["policy_preds"]
    after = motion["policy_preds"]

    total_before += len(before)
    total_after += len(after)

    # Quand une box est supprimée par clipping,
    # l'appariement par indice n'est plus garanti.
    if len(before) != len(after):
        continue

    for a, b in zip(before, after):

        ax = (a[0] + a[2]) / 2
        ay = (a[1] + a[3]) / 2

        bx = (b[0] + b[2]) / 2
        by = (b[1] + b[3]) / 2

        dist = float(np.hypot(bx - ax, by - ay))
        displacements.append(dist)

        if dist > 0.01:
            moved += 1
        else:
            same += 1

print("\n========== MOUVEMENT DES BOXES ==========")

print("Boîtes Direct Reuse :", total_before)
print("Boîtes Motion Compensation :", total_after)
print("Boîtes déplacées comparables :", moved)
print("Boîtes inchangées comparables :", same)

if displacements:
    print(
        "Déplacement médian (pixels) :",
        round(float(np.median(displacements)), 3)
    )
    print(
        "Déplacement P95 (pixels) :",
        round(float(np.percentile(displacements, 95)), 3)
    )

print("\n========== CONCLUSION DU DIAGNOSTIC ==========")

delta_map = (
    diagnostic_df.iloc[1]["mAP50-95 (%)"]
    - diagnostic_df.iloc[0]["mAP50-95 (%)"]
)

print(
    "Variation mAP50:95 sur frames réutilisées :",
    round(float(delta_map), 3),
    "points"
)

if delta_map < 0:
    print(
        "La compensation dégrade la localisation "
        "sur les frames annotées réutilisées."
    )
elif delta_map > 0:
    print(
        "La compensation améliore la localisation "
        "sur les frames annotées réutilisées."
    )
else:
    print("Aucun changement détecté sur ce sous-ensemble.")


========== DISTRIBUTION ==========
Frames annotées : 9982
Frames fraîches : 4813
Frames réutilisées : 5169
Réutilisation annotée (%) : 51.783
Prédictions fraîches identiques : True

========== FRAMES RÉUTILISÉES UNIQUEMENT ==========


,Méthode,Frames GT,mAP50 (%),mAP50-95 (%),Precision@50 (%),Recall@50 (%),Recall@75 (%),TP@50,FP@50,FN@50
0,Direct Reuse,5169,40.528,21.547,79.764,50.939,32.799,25660,6510,24714
1,Motion Compensation,5169,40.284,21.162,79.064,50.490,31.866,25434,6735,24940



========== MOUVEMENT DES BOXES ==========
Boîtes Direct Reuse : 775130
Boîtes Motion Compensation : 775127
Boîtes déplacées comparables : 29045
Boîtes inchangées comparables : 745770
Déplacement médian (pixels) : 0.0
Déplacement P95 (pixels) : 0.0

========== CONCLUSION DU DIAGNOSTIC ==========
Variation mAP50:95 sur frames réutilisées : -0.385 points
La compensation dégrade la localisation sur les frames annotées réutilisées.
